# 02 - Age-dependent DNA methylation of genes that are suppressed in stem cells is a hallmark of cancer - Teschendorff's study

In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     02-age-dependent-dna-teschendorff                  ║
# ║ Description:  Age-dependent DNA - Teschendorff's extention       ║
# ║ Dataset(s):   GSE225845                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 01-Feb-2026 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [1]:
# Latex
!sudo apt-get update -qq
!sudo apt-get install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cm-super-minimal dvisvgm fonts-lato fonts-lmodern fonts-texgyre
  libapache-pom-java libcommons-logging-java libcommons-parent-java
  libfontbox-java libkpathsea6 libpdfbox-java libptexenc1 libruby3.0
  libsynctex2 libteckit0 libtexlua53 libtexluajit2 libwoff1 libzzip-0-13
  lmodern pfb2t1c2pfb preview-latex-style rake ruby ruby-net-telnet
  ruby-rubygems ruby-webrick ruby-xmlrpc ruby3.0 rubygems-integration t1utils
  tex-common tex-gyre texlive-base texlive-binaries texlive-latex-base
  texlive-latex-recommended texlive-pictures texlive-plain-generic tipa
  xfonts-encodings xfonts-utils
Suggested packages:
  libavalon-framework-java libcommons-loggin

In [2]:
# Import
import os
import numpy as np
import pandas as pd
import polars as pl

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# THESIS STYLE FOR PLOT
def apply_thesis_style(use_tex: bool = True,
                       legend_position: str = "upper right",
                       legend_outside: bool = False):
    """
    Applies a uniform style (Matplotlib + Seaborn) consistent with LaTeX
    and defines `place_legend()` to position legends consistently.

    Parameters
    ----------
    use_tex : bool
        If True, enables LaTeX text rendering (requires TeX installed).
    legend_position : str
        DEFAULT position for legends: 'upper right', 'upper left',
        'lower right', 'lower left', 'center', 'best' (also accepts 'top/bottom').
    legend_outside : bool
        If True, the default legend is outside the plot (to the right).
    """

    # --- Normalize default position ---
    def _normalize_pos(pos: str) -> str:
        if not isinstance(pos, str):
            return "upper right"
        key = pos.strip().lower().replace("top", "upper").replace("bottom", "lower")
        mapping = {
            "upper right": "upper right",
            "upper left":  "upper left",
            "lower right": "lower right",
            "lower left":  "lower left",
            "center":      "center",
            "best":        "best",
        }
        return mapping.get(key, "upper right")

    _default_loc = _normalize_pos(legend_position)

    # --- Seaborn theme + rcParams consistent with thesis ---
    sns.set_theme(style="whitegrid", context="notebook")
    mpl.rcParams.update({
        # Typography
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman", "Latin Modern Roman", "Times New Roman"],
        "mathtext.fontset": "cm",
        "text.usetex": bool(use_tex),

        # Sizes
        "figure.figsize": (6.8, 4.5),
        "font.size": 8.5,          # generic text (plt.text, etc.)
        "axes.labelsize": 10.5,    # axis labels
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9.5,
        "legend.title_fontsize": 9.5,

        # Axes look
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.facecolor": "white",
        "axes.edgecolor": "#D0D0D0",
        "axes.linewidth": 0.8,

        # Legend style (default; can be overridden by place_legend)
        "legend.frameon": True,
        "legend.facecolor": "white",
        "legend.edgecolor": "#D0D0D0",
        "legend.loc": _default_loc,
        "legend.framealpha": 1.0,
        "legend.handlelength": 1.8,
        "legend.handletextpad": 0.6,
        "legend.borderpad": 0.4,
        "legend.borderaxespad": 0.8,
    })

    mpl.rcParams["image.cmap"] = "viridis"
    sns.set_palette("viridis")

    def place_legend(ax=None,
                     position: str | None = None,
                     outside: bool | None = None,
                     fontsize: float | None = None):
        """
        Places the legend on an axis with consistent thesis style.

        Parameters
        ----------
        ax : matplotlib.axes.Axes, optional
            Axis on which to place the legend (default: current axis).
        position : str | None
            'upper right', 'upper left', 'lower right', 'lower left', 'center', 'best'.
            If None, uses the default position passed to apply_thesis_style().
            Spaces/caps tolerated; 'top/bottom' mapped to 'upper/lower'.
        outside : bool | None
            If True, places the legend outside the plot (to the right).
            If None, uses the default value passed to apply_thesis_style().
        fontsize : float | None
            To change the legend font size only in this plot.
        """

        if ax is None:
            ax = plt.gca()

        # Normalize requested position or use global default
        loc_val = _default_loc
        if isinstance(position, str):
            key = position.strip().lower().replace("top", "upper").replace("bottom", "lower")
            loc_map = {
                "upper right": "upper right",
                "upper left":  "upper left",
                "lower right": "lower right",
                "lower left":  "lower left",
                "center":      "center",
                "best":        "best",
            }
            loc_val = loc_map.get(key, _default_loc)

        # outside: if None, inherit from default; otherwise use override
        outside = (legend_outside if outside is None else bool(outside))

        # Build kwargs consistent with rcParams
        legend_kwargs = dict(
            loc=loc_val,
            frameon=True,
            facecolor="white",
            edgecolor=mpl.rcParams["legend.edgecolor"],
            framealpha=1.0,
            fontsize=mpl.rcParams["legend.fontsize"] if fontsize is None else fontsize,
            handlelength=1.8,
            handletextpad=0.6,
            borderpad=0.4,
            fancybox=False,
        )

        if outside:
            legend_kwargs.update({
                "bbox_to_anchor": (1.02, 1.0),
                "borderaxespad": 0.0,
            })

        leg = ax.legend(**legend_kwargs)
        if leg is not None:
            leg.get_frame().set_linewidth(0.8)
            leg.get_frame().set_edgecolor(mpl.rcParams["legend.edgecolor"])
            leg.get_frame().set_facecolor("white")
        return ax

    # Export the helper to the global notebook/script namespace
    globals()["place_legend"] = place_legend


apply_thesis_style(use_tex=True)

## 01- Study on real data

In [6]:
RUN1_OUTDIR = "./gse225845_run1_age_true_outputs_optimized"
RUN2_OUTDIR = "./gse225845_run2_age_horvath_outputs_optimized"


In [7]:
# CHECK AGE EFFECT IN NORMAL SAMPLES (CHRONOLOGICAL AGE)
# Analysis restricted to Normal (0) and Adjacent (1) samples;
# Tumor samples are excluded from the age-effect assessment.

# CONFIG
BETA_PARQUET_PATH  = "/kaggle/input/gse225845-parquet/GSE225845.parquet"
PHENO_PARQUET_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"

ID_COL   = "id_tissue"
LABEL_COL = "label"
AGE_COL  = "age_at_surgery"

NORMAL_LABEL = 0

TOP_K_FOR_PCA = 10000
N_PCS = 5
RANDOM_STATE = 0
MIN_OBS_NORMAL = 20

OUTDIR = RUN1_OUTDIR
os.makedirs(OUTDIR, exist_ok=True)


# Helpers
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=np.float64)
    q = np.full_like(p, np.nan, dtype=np.float64)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    p_ok = p[ok]
    n = p_ok.size
    order = np.argsort(p_ok)
    ranked = p_ok[order]
    q_ok = ranked * n / (np.arange(1, n + 1))
    q_ok = np.minimum.accumulate(q_ok[::-1])[::-1]
    q_ok = np.clip(q_ok, 0.0, 1.0)
    out = np.empty_like(q_ok)
    out[order] = q_ok
    q[ok] = out
    return q

def corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.nanmean(a)
    b = b - np.nanmean(b)
    denom = np.sqrt(np.nansum(a*a) * np.nansum(b*b))
    return np.nan if (denom == 0 or np.isnan(denom)) else float(np.nansum(a*b) / denom)

def fit_pca_on_normal_and_project(X_all: np.ndarray, normal_mask: np.ndarray, n_pcs: int):
    X_norm = X_all[normal_mask, :]
    means = np.nanmean(X_norm, axis=0)
    X_norm = np.where(np.isnan(X_norm), means, X_norm)
    X_all  = np.where(np.isnan(X_all),  means, X_all)

    scaler = StandardScaler(with_mean=True, with_std=True)
    X_norm_z = scaler.fit_transform(X_norm)
    X_all_z  = scaler.transform(X_all)

    pca = PCA(n_components=n_pcs, random_state=RANDOM_STATE)
    pcs_norm = pca.fit_transform(X_norm_z)
    pcs_all  = pca.transform(X_all_z)
    return pcs_norm, pcs_all


# 1) LOAD + JOIN (polars)
pheno = pl.read_parquet(PHENO_PARQUET_PATH)

# keep only needed cols
need_pheno = [ID_COL, LABEL_COL, AGE_COL]
missing = [c for c in need_pheno if c not in pheno.columns]
if missing:
    raise ValueError(f"Missing in pheno: {missing}. Available: {pheno.columns}")

pheno = pheno.select(need_pheno).with_columns([
    pl.col(ID_COL).cast(pl.Utf8),
    pl.col(LABEL_COL).cast(pl.Int64),
    pl.col(AGE_COL).cast(pl.Float64),
]).drop_nulls([AGE_COL])

beta = pl.read_parquet(BETA_PARQUET_PATH)

# check beta format
beta_cols = beta.columns
has_id = ID_COL in beta_cols
has_label = LABEL_COL in beta_cols

if not (has_id and has_label):
    raise ValueError(
        f"Beta must contain '{ID_COL}' and '{LABEL_COL}' columns per your statement. "
        f"Found columns: {beta_cols[:30]}"
    )

# Identify CpG columns (anything starting with 'cg') for wide format
cpg_cols = [c for c in beta_cols if isinstance(c, str) and c.startswith("cg")]

if len(cpg_cols) > 1000:
    # ---- CASE A: wide beta: rows=samples, columns=CpGs
    print("Detected WIDE beta format (rows=samples, columns=CpGs).")

    beta = beta.select([ID_COL, LABEL_COL] + cpg_cols).with_columns([
        pl.col(ID_COL).cast(pl.Utf8),
        pl.col(LABEL_COL).cast(pl.Int64),
    ])

    # join pheno -> keep only samples with age
    df = beta.join(pheno, on=[ID_COL, LABEL_COL], how="inner")
    # keep only Normal (0) and Adjacent (1)
    df = df.filter(pl.col(LABEL_COL).is_in([0, 1]))


    # to pandas in a controlled way
    df_pd = df.to_pandas()
    # keep only Normal (0) and Adjacent (1)
    df_pd = df_pd[df_pd[LABEL_COL].isin([0, 1])].reset_index(drop=True)


    # sample axis
    samples = df_pd[ID_COL].astype(str).to_list()
    labels  = df_pd[LABEL_COL].astype(int).to_numpy()
    ages    = df_pd[AGE_COL].astype(float).to_numpy()

    # build CpG x samples matrix
    beta_mat = df_pd[cpg_cols].to_numpy(dtype=np.float32).T  # CpGs x samples
    cpg_index = np.array(cpg_cols, dtype=object)

elif ("cpg" in beta_cols) and ("beta" in beta_cols):
    # ---- CASE B: long beta: columns id_tissue, label, cpg, beta
    print("Detected LONG beta format (id_tissue,label,cpg,beta) — pivoting.")

    beta = beta.select([ID_COL, LABEL_COL, "cpg", "beta"]).with_columns([
        pl.col(ID_COL).cast(pl.Utf8),
        pl.col(LABEL_COL).cast(pl.Int64),
        pl.col("cpg").cast(pl.Utf8),
        pl.col("beta").cast(pl.Float32),
    ])

    # join pheno (age) at sample-level
    df = beta.join(pheno, on=[ID_COL, LABEL_COL], how="inner")

    # pivot to wide: rows=cpg, columns=id_tissue
    wide = df.pivot(values="beta", index="cpg", columns=ID_COL, aggregate_function="first")

    wide_pd = wide.to_pandas()
    cpg_index = wide_pd["cpg"].astype(str).to_numpy()
    wide_pd = wide_pd.set_index("cpg")

    # keep sample ordering and matching arrays
    samples = list(wide_pd.columns)
    # recover labels/ages per sample from joined df (unique per id_tissue)
    meta = df.select([ID_COL, LABEL_COL, AGE_COL]).unique().to_pandas().set_index(ID_COL)
    labels = meta.loc[samples, LABEL_COL].astype(int).to_numpy()
    ages   = meta.loc[samples, AGE_COL].astype(float).to_numpy()

    beta_mat = wide_pd.to_numpy(dtype=np.float32)  # CpGs x samples

else:
    raise ValueError(
        "Could not infer beta format. "
        "Expected either many 'cg...' columns (wide) or columns ['cpg','beta'] (long). "
        f"First columns: {beta_cols[:30]}"
    )

print("Aligned samples:", len(samples))
print("Beta matrix shape (CpGs x samples):", beta_mat.shape)

normal_mask = (labels == NORMAL_LABEL)
if normal_mask.sum() < max(10, MIN_OBS_NORMAL):
    raise ValueError(f"Too few Normal samples ({normal_mask.sum()}). Check NORMAL_LABEL and labels distribution.")

# 2) Quick age-by-label plot
apply_thesis_style(use_tex=True, legend_position="best", legend_outside=False)

tmp = pd.DataFrame({
    "label": labels,
    "age_at_surgery": ages
})

palette_labels = sns.color_palette("viridis", n_colors=2)

plt.figure()
sns.boxplot(
    data=tmp,
    x="label",
    y="age_at_surgery",
    palette=palette_labels,
    linewidth=0.8,
    fliersize=3
)
plt.xlabel("Label")
plt.ylabel("Age at surgery (years)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "age_by_label.pdf"))
plt.close()


# 3) TOP-K variable CpGs in Normal (for PCA)
beta_norm = beta_mat[:, normal_mask]
var_norm = np.nanvar(beta_norm, axis=1).astype(np.float64)
top_k = min(TOP_K_FOR_PCA, beta_mat.shape[0])
top_idx = np.argsort(var_norm)[::-1][:top_k]

X_all_raw = beta_mat[top_idx, :].T  # samples x top_k

# 4) PCA RAW
pcs_norm_raw, pcs_all_raw = fit_pca_on_normal_and_project(X_all_raw, normal_mask, N_PCS)
corr_pc1_age_raw = corr(pcs_norm_raw[:, 0], ages[normal_mask])

emb_raw = pd.DataFrame(pcs_all_raw, columns=[f"PC{i+1}_raw" for i in range(N_PCS)])
emb_raw[ID_COL] = samples
emb_raw["label"] = labels
emb_raw["age_at_surgery"] = ages
emb_raw.to_csv(os.path.join(OUTDIR, "pca_raw_embeddings.csv"), index=False)

emb_raw_plot = emb_raw.rename(columns={
    "label": "Label",
    "age_at_surgery": "Age at surgery"
})

apply_thesis_style(use_tex=True, legend_position="best", legend_outside=True)

palette_labels = sns.color_palette("viridis", n_colors=2)

plt.figure()
sns.scatterplot(
    data=emb_raw,
    x="PC1_raw",
    y="PC2_raw",
    hue="label",
    palette=palette_labels,
    size="age_at_surgery",
    sizes=(20, 80),
    alpha=0.85,
    edgecolor="none"
)
plt.xlabel("PC1 (raw)")
plt.ylabel("PC2 (raw)")
place_legend(outside=True)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "pca_raw.pdf"))
plt.close()
plt.close()

print(f"GSE225845 — PCA raw (fit Normal), corr(PC1, age)={corr_pc1_age_raw:.2f}")

# 5) CpG-wise age association in Normal (SUMS-based OLS)
Y = beta_norm.astype(np.float32)            # CpGs x n_norm
x = ages[normal_mask].astype(np.float64)    # n_norm
x2 = x * x

obs = ~np.isnan(Y)
n_obs = obs.sum(axis=1).astype(np.int64)
valid = n_obs >= MIN_OBS_NORMAL

sy  = np.nansum(Y, axis=1).astype(np.float64)
sy2 = np.nansum((Y * Y), axis=1).astype(np.float64)
sxy = np.nansum(Y * x[None, :], axis=1).astype(np.float64)
sx  = (obs * x[None, :]).sum(axis=1).astype(np.float64)
sx2 = (obs * x2[None, :]).sum(axis=1).astype(np.float64)

den = sx2 - (sx * sx) / np.maximum(n_obs, 1)
num = sxy - (sx * sy) / np.maximum(n_obs, 1)

b = np.full(Y.shape[0], np.nan, dtype=np.float64)
a = np.full(Y.shape[0], np.nan, dtype=np.float64)

good = valid & np.isfinite(den) & (den > 0)
b[good] = num[good] / den[good]
a[good] = (sy[good] - b[good] * sx[good]) / n_obs[good]

# p-values
df = (n_obs - 2).astype(np.float64)
# fast approx (normal) if scipy missing
t_stat = np.full(Y.shape[0], np.nan, dtype=np.float64)
# SSE via sums
SSE = np.full(Y.shape[0], np.nan, dtype=np.float64)
SSE[good] = (
    sy2[good]
    - 2.0 * a[good] * sy[good]
    - 2.0 * b[good] * sxy[good]
    + n_obs[good] * (a[good] ** 2)
    + 2.0 * a[good] * b[good] * sx[good]
    + (b[good] ** 2) * sx2[good]
)
sigma2 = np.full(Y.shape[0], np.nan, dtype=np.float64)
sigma2[good] = SSE[good] / df[good]
se_b = np.full(Y.shape[0], np.nan, dtype=np.float64)
se_b[good] = np.sqrt(sigma2[good] / den[good])
t_stat[good] = b[good] / se_b[good]

try:
    from scipy.stats import t as tdist
    pvals = np.full(Y.shape[0], np.nan, dtype=np.float64)
    pvals[good] = 2.0 * tdist.sf(np.abs(t_stat[good]), df=df[good])
except Exception:
    # normal approximation
    pvals = np.full(Y.shape[0], np.nan, dtype=np.float64)
    x_ = np.abs(t_stat[good]) / np.sqrt(2.0)
    Phi = 0.5 * (1.0 + np.erf(x_))
    pvals[good] = 2.0 * (1.0 - Phi)

qvals = bh_fdr(pvals)

assoc = pd.DataFrame({
    "cpg": cpg_index,
    "n_obs_normal": n_obs,
    "slope_per_year": b,
    "intercept": a,
    "t_stat": t_stat,
    "pval": pvals,
    "qval_fdr": qvals
}).set_index("cpg")
assoc.to_csv(os.path.join(OUTDIR, "age_association_normal_cpgwise.csv"))

# 6) Residualization ONLY for PCA features
a_top = a[top_idx].astype(np.float32)
b_top = b[top_idx].astype(np.float32)
ages_f = ages.astype(np.float32)

X_all_resid = X_all_raw.astype(np.float32) - a_top[None, :] - ages_f[:, None] * b_top[None, :]

# 7) PCA residualized
pcs_norm_res, pcs_all_res = fit_pca_on_normal_and_project(X_all_resid, normal_mask, N_PCS)
corr_pc1_age_res = corr(pcs_norm_res[:, 0], ages[normal_mask])

emb_res = pd.DataFrame(pcs_all_res, columns=[f"PC{i+1}_resid" for i in range(N_PCS)])
emb_res[ID_COL] = samples
emb_res["label"] = labels
emb_res["age_at_surgery"] = ages
emb_res.to_csv(os.path.join(OUTDIR, "pca_residual_embeddings.csv"), index=False)

apply_thesis_style(use_tex=True, legend_position="best", legend_outside=True)

plt.figure()
sns.scatterplot(
    data=emb_res,
    x="PC1_resid",
    y="PC2_resid",
    hue="label",
    palette=palette_labels,
    size="age_at_surgery",
    sizes=(20, 80),
    alpha=0.85,
    edgecolor="none"
)
plt.xlabel("PC1 (residualized)")
plt.ylabel("PC2 (residualized)")
place_legend(outside=True)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "pca_residual.pdf"))
plt.close()

print(f"GSE225845 — PCA residualized, corr(PC1, age)={corr_pc1_age_res:.2f}")

# 8) Summary
n_age_cpg = int((assoc["qval_fdr"] < 0.05).sum())
summary = pd.DataFrame([{
    "corr_PC1_raw_age_in_Normal": corr_pc1_age_raw,
    "corr_PC1_resid_age_in_Normal": corr_pc1_age_res,
    "n_samples_total": int(len(samples)),
    "n_samples_normal": int(normal_mask.sum()),
    "n_cpgs_total": int(beta_mat.shape[0]),
    "top_k_for_pca": int(top_k),
    "min_obs_normal_for_ols": int(MIN_OBS_NORMAL),
    "n_age_cpgs_FDR_0p05": n_age_cpg,
}])
summary.to_csv(os.path.join(OUTDIR, "run1_summary_metrics.csv"), index=False)

print("Saved outputs in:", OUTDIR)
print(summary.to_string(index=False))


Detected WIDE beta format (rows=samples, columns=CpGs).
Aligned samples: 253
Beta matrix shape (CpGs x samples): (747602, 253)


/tmp/ipykernel_54/1565390472.py:183: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


GSE225845 — PCA raw (fit Normal), corr(PC1, age)=-0.03
GSE225845 — PCA residualized, corr(PC1, age)=0.00
Saved outputs in: ./gse225845_run1_age_true_outputs_optimized
 corr_PC1_raw_age_in_Normal  corr_PC1_resid_age_in_Normal  n_samples_total  n_samples_normal  n_cpgs_total  top_k_for_pca  min_obs_normal_for_ols  n_age_cpgs_FDR_0p05
                  -0.033114                  3.897190e-08              253               113        747602          10000                      20                  533


In [8]:
# SUMMARY OF AGE EFFECT ANALYSIS (CHRONOLOGICAL AGE, NORMAL-ONLY)
OUTDIR = "./gse225845_run1_age_true_outputs_optimized"

RAW_PATH   = os.path.join(OUTDIR, "pca_raw_embeddings.csv")
RES_PATH   = os.path.join(OUTDIR, "pca_residual_embeddings.csv")
ASSOC_PATH = os.path.join(OUTDIR, "age_association_normal_cpgwise.csv")

# Helpers
def corr_nan(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return np.nan
    a = a[ok] - np.mean(a[ok])
    b = b[ok] - np.mean(b[ok])
    denom = np.sqrt(np.sum(a*a) * np.sum(b*b))
    return np.nan if denom == 0 else float(np.sum(a*b) / denom)

def summarize_age_cpgs(assoc_df, q_col="qval_fdr"):
    qs = assoc_df[q_col].to_numpy(dtype=np.float64)
    out = {}
    for thr in [0.20, 0.10, 0.05, 0.01]:
        out[f"n_age_cpgs_{q_col}_lt_{thr}"] = int(np.nansum(qs < thr))
    return out

# Load
emb_raw = pd.read_csv(RAW_PATH)
emb_res = pd.read_csv(RES_PATH)
assoc   = pd.read_csv(ASSOC_PATH)

# sanity: columns
assert "age_at_surgery" in emb_raw.columns, "age_at_surgery missing in pca_raw_embeddings.csv"
assert "age_at_surgery" in emb_res.columns, "age_at_surgery missing in pca_residual_embeddings.csv"
assert "label" in emb_raw.columns and "label" in emb_res.columns, "label missing in embeddings"
assert "qval_fdr" in assoc.columns, "qval_fdr missing in age_association file"
assert "slope_per_year" in assoc.columns, "slope_per_year missing in age_association file"

# if assoc has CpG in index or column
if "cpg" in assoc.columns:
    assoc = assoc.set_index("cpg")
else:
    # if already saved with index, keep as-is
    if assoc.index.name is None:
        assoc.index.name = "cpg"

# 1 How many Age-CpGs 
counts = summarize_age_cpgs(assoc, q_col="qval_fdr")
print("=== Age-CpG counts (Normal-only association) ===")
for k, v in counts.items():
    print(f"{k}: {v}")

# 2 corr(age, PCs) in NORMAL 
NORMAL_LABEL = 0
raw_norm = emb_raw.loc[emb_raw["label"] == NORMAL_LABEL].copy()
res_norm = emb_res.loc[emb_res["label"] == NORMAL_LABEL].copy()

pc_cols_raw = [c for c in raw_norm.columns if c.startswith("PC") and c.endswith("_raw")]
pc_cols_res = [c for c in res_norm.columns if c.startswith("PC") and c.endswith("_resid")]

pc_cols_raw = sorted(pc_cols_raw, key=lambda s: int(s.split("PC")[1].split("_")[0]))
pc_cols_res = sorted(pc_cols_res, key=lambda s: int(s.split("PC")[1].split("_")[0]))

max_pcs = min(5, len(pc_cols_raw), len(pc_cols_res))

corr_rows = []
for i in range(max_pcs):
    pc_r = pc_cols_raw[i]
    pc_e = pc_cols_res[i]
    corr_rows.append({
        "PC": f"PC{i+1}",
        "corr_raw_age_in_Normal": corr_nan(raw_norm[pc_r], raw_norm["age_at_surgery"]),
        "corr_resid_age_in_Normal": corr_nan(res_norm[pc_e], res_norm["age_at_surgery"]),
    })

corr_df = pd.DataFrame(corr_rows)
print("\n=== Correlation between age and PCs (Normal only) ===")
print(corr_df.to_string(index=False))

corr_df.to_csv(os.path.join(OUTDIR, "corr_age_vs_pcs_normal.csv"), index=False)

# 3 Effect size summaries for Age-CpGs 
age_cpgs_05 = assoc.loc[assoc["qval_fdr"] < 0.05].copy()
print("\n=== Effect sizes (slope per year) for Age-CpGs (FDR<0.05) ===")
if age_cpgs_05.shape[0] == 0:
    print("No CpGs at FDR<0.05.")
else:
    slopes = age_cpgs_05["slope_per_year"].to_numpy(dtype=np.float64)
    abs_slopes = np.abs(slopes[np.isfinite(slopes)])
    print("n:", abs_slopes.size)
    print("abs(slope) quantiles (beta units / year):",
          np.quantile(abs_slopes, [0.50, 0.75, 0.90, 0.95, 0.99]))
    print("max abs(slope):", float(np.max(abs_slopes)))

# Save a compact table of top Age-CpGs by |t| (or by smallest q)
top_by_q = assoc.sort_values("qval_fdr", ascending=True).head(200)
top_by_q.to_csv(os.path.join(OUTDIR, "top200_age_cpgs_by_qval.csv"))

top_by_abs_t = assoc.assign(abs_t=np.abs(assoc["t_stat"])).sort_values("abs_t", ascending=False).head(200)
top_by_abs_t.drop(columns=["abs_t"]).to_csv(os.path.join(OUTDIR, "top200_age_cpgs_by_abs_t.csv"))

# 5 One-line "result" string you can paste into notes 
result_line = (
    f"GSE225845 (Normal-only fit): corr(age, PC1) raw={corr_df.loc[0,'corr_raw_age_in_Normal']:.3f}, "
    f"resid={corr_df.loc[0,'corr_resid_age_in_Normal']:.3f}; "
    f"Age-CpGs (FDR<0.05)={counts['n_age_cpgs_qval_fdr_lt_0.05']}"
)
print("\n=== One-line summary ===")
print(result_line)
with open(os.path.join(OUTDIR, "run1_one_line_summary.txt"), "w") as f:
    f.write(result_line + "\n")


=== Age-CpG counts (Normal-only association) ===
n_age_cpgs_qval_fdr_lt_0.2: 2873
n_age_cpgs_qval_fdr_lt_0.1: 962
n_age_cpgs_qval_fdr_lt_0.05: 533
n_age_cpgs_qval_fdr_lt_0.01: 217

=== Correlation between age and PCs (Normal only) ===
 PC  corr_raw_age_in_Normal  corr_resid_age_in_Normal
PC1               -0.033114              4.360097e-08
PC2                0.001541             -4.723215e-08
PC3               -0.070863             -9.994525e-08
PC4               -0.242033              5.616517e-08
PC5               -0.012772              2.234485e-07

=== Effect sizes (slope per year) for Age-CpGs (FDR<0.05) ===
n: 533
abs(slope) quantiles (beta units / year): [0.00132439 0.00203426 0.00267613 0.00304139 0.0038901 ]
max abs(slope): 0.0045991568225476

=== One-line summary ===
GSE225845 (Normal-only fit): corr(age, PC1) raw=-0.033, resid=0.000; Age-CpGs (FDR<0.05)=533


## 02- Study on Horvath's age

In [9]:
# CHECK AGE EFFECT IN NORMAL SAMPLES (HORVATH DNAmAge)
# Analysis restricted to Normal (0) and Adjacent (1) samples;
# Tumor samples are excluded from the age-effect assessment.

# CONFIG
BETA_PARQUET_PATH  = "/kaggle/input/gse225845-parquet/GSE225845.parquet"
HORVATH_AGE        = "/kaggle/input/gse225845-horvathdnamage/Output_GSE225845_HorvathDNAmAge_normal_adj_only.csv"

ID_COL    = "id_tissue"
LABEL_COL = "label"
AGE_COL   = "DNAmAge"          # <-- Horvath age column name in the CSV

NORMAL_LABEL = 0

TOP_K_FOR_PCA = 10000
N_PCS = 5
RANDOM_STATE = 0
MIN_OBS_NORMAL = 20

OUTDIR = RUN2_OUTDIR
os.makedirs(OUTDIR, exist_ok=True)


# Helpers
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=np.float64)
    q = np.full_like(p, np.nan, dtype=np.float64)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    p_ok = p[ok]
    n = p_ok.size
    order = np.argsort(p_ok)
    ranked = p_ok[order]
    q_ok = ranked * n / (np.arange(1, n + 1))
    q_ok = np.minimum.accumulate(q_ok[::-1])[::-1]
    q_ok = np.clip(q_ok, 0.0, 1.0)
    out = np.empty_like(q_ok)
    out[order] = q_ok
    q[ok] = out
    return q

def corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.nanmean(a)
    b = b - np.nanmean(b)
    denom = np.sqrt(np.nansum(a*a) * np.nansum(b*b))
    return np.nan if (denom == 0 or np.isnan(denom)) else float(np.nansum(a*b) / denom)

def fit_pca_on_normal_and_project(X_all: np.ndarray, normal_mask: np.ndarray, n_pcs: int):
    X_norm = X_all[normal_mask, :]
    means = np.nanmean(X_norm, axis=0)
    X_norm = np.where(np.isnan(X_norm), means, X_norm)
    X_all  = np.where(np.isnan(X_all),  means, X_all)

    scaler = StandardScaler(with_mean=True, with_std=True)
    X_norm_z = scaler.fit_transform(X_norm)
    X_all_z  = scaler.transform(X_all)

    pca = PCA(n_components=n_pcs, random_state=RANDOM_STATE)
    pcs_norm = pca.fit_transform(X_norm_z)
    pcs_all  = pca.transform(X_all_z)
    return pcs_norm, pcs_all


# 1) LOAD Horvath age (CSV) + beta (parquet)
age_df = pl.read_csv(HORVATH_AGE)

# Sanity for Horvath file
if ("SampleID" not in age_df.columns) or (AGE_COL not in age_df.columns):
    raise ValueError(f"Horvath file must contain columns ['SampleID','{AGE_COL}']. Found: {age_df.columns}")

# Extract GSM id to match beta id_tissue robustly
# Example: SampleID=GSM8744013 -> sample_base=GSM8744013
age_df = age_df.with_columns([
    pl.col("SampleID").cast(pl.Utf8),
    pl.col(AGE_COL).cast(pl.Float64),
    pl.col("SampleID").str.extract(r"(GSM\d+)", 1).alias("sample_base")
]).drop_nulls(["sample_base", AGE_COL])

beta = pl.read_parquet(BETA_PARQUET_PATH)

# check beta format
beta_cols = beta.columns
has_id = ID_COL in beta_cols
has_label = LABEL_COL in beta_cols

if not (has_id and has_label):
    raise ValueError(
        f"Beta must contain '{ID_COL}' and '{LABEL_COL}' columns. "
        f"Found columns: {beta_cols[:30]}"
    )

# Identify CpG columns (anything starting with 'cg') for wide format
cpg_cols = [c for c in beta_cols if isinstance(c, str) and c.startswith("cg")]

# Add sample_base extracted from id_tissue (e.g., 'GSM8744013_0' -> 'GSM8744013')
beta = beta.with_columns([
    pl.col(ID_COL).cast(pl.Utf8),
    pl.col(LABEL_COL).cast(pl.Int64),
    pl.col(ID_COL).str.extract(r"(GSM\d+)", 1).alias("sample_base")
])

# Require that sample_base exists (otherwise join is impossible)
if beta.select(pl.col("sample_base").is_null().sum()).item() > 0:
    print("[Warning] Some id_tissue values do not contain a 'GSM#######' pattern. "
          "Those samples will be dropped when joining with Horvath ages.")

if len(cpg_cols) > 1000:
    # ---- CASE A: wide beta: rows=samples, columns=CpGs
    print("Detected WIDE beta format (rows=samples, columns=CpGs).")

    beta = beta.select([ID_COL, LABEL_COL, "sample_base"] + cpg_cols)

    # join Horvath age at sample-level using sample_base
    df = beta.join(age_df.select(["sample_base", AGE_COL]), on="sample_base", how="inner")

    # keep only Normal (0) and Adjacent (1)
    df = df.filter(pl.col(LABEL_COL).is_in([0, 1]))

    # to pandas in a controlled way
    df_pd = df.to_pandas().reset_index(drop=True)

    # sample axis
    samples = df_pd[ID_COL].astype(str).to_list()
    labels  = df_pd[LABEL_COL].astype(int).to_numpy()
    ages    = df_pd[AGE_COL].astype(float).to_numpy()

    # build CpG x samples matrix
    beta_mat = df_pd[cpg_cols].to_numpy(dtype=np.float32).T  # CpGs x samples
    cpg_index = np.array(cpg_cols, dtype=object)

elif ("cpg" in beta_cols) and ("beta" in beta_cols):
    # ---- CASE B: long beta: columns id_tissue, label, cpg, beta
    print("Detected LONG beta format (id_tissue,label,cpg,beta) — pivoting.")

    beta = beta.select([ID_COL, LABEL_COL, "sample_base", "cpg", "beta"]).with_columns([
        pl.col("cpg").cast(pl.Utf8),
        pl.col("beta").cast(pl.Float32),
    ])

    # join Horvath age at sample-level
    df = beta.join(age_df.select(["sample_base", AGE_COL]), on="sample_base", how="inner")

    # keep only Normal (0) and Adjacent (1)
    df = df.filter(pl.col(LABEL_COL).is_in([0, 1]))

    # pivot to wide: rows=cpg, columns=id_tissue
    wide = df.pivot(values="beta", index="cpg", columns=ID_COL, aggregate_function="first")

    wide_pd = wide.to_pandas()
    cpg_index = wide_pd["cpg"].astype(str).to_numpy()
    wide_pd = wide_pd.set_index("cpg")

    # keep sample ordering and matching arrays
    samples = list(wide_pd.columns)

    # recover labels/ages per sample from df (unique per id_tissue)
    meta = df.select([ID_COL, LABEL_COL, AGE_COL]).unique().to_pandas().set_index(ID_COL)
    labels = meta.loc[samples, LABEL_COL].astype(int).to_numpy()
    ages   = meta.loc[samples, AGE_COL].astype(float).to_numpy()

    beta_mat = wide_pd.to_numpy(dtype=np.float32)  # CpGs x samples

else:
    raise ValueError(
        "Could not infer beta format. "
        "Expected either many 'cg...' columns (wide) or columns ['cpg','beta'] (long). "
        f"First columns: {beta_cols[:30]}"
    )

print("Aligned samples:", len(samples))
print("Beta matrix shape (CpGs x samples):", beta_mat.shape)

normal_mask = (labels == NORMAL_LABEL)
if normal_mask.sum() < max(10, MIN_OBS_NORMAL):
    raise ValueError(f"Too few Normal samples ({normal_mask.sum()}). Check NORMAL_LABEL and labels distribution.")

# 2) Quick age-by-label plot
apply_thesis_style(use_tex=True, legend_position="best", legend_outside=False)

tmp = pd.DataFrame({
    "label": labels,
    AGE_COL: ages
})

palette_labels = sns.color_palette("viridis", n_colors=2)

plt.figure()
sns.boxplot(
    data=tmp,
    x="label",
    y=AGE_COL,
    palette=palette_labels,
    linewidth=0.8,
    fliersize=3
)
plt.xlabel("Label")
plt.ylabel("Horvath DNAmAge (years)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "age_by_label.pdf"))
plt.close()

# 3) TOP-K variable CpGs in Normal (for PCA)
beta_norm = beta_mat[:, normal_mask]
var_norm = np.nanvar(beta_norm, axis=1).astype(np.float64)
top_k = min(TOP_K_FOR_PCA, beta_mat.shape[0])
top_idx = np.argsort(var_norm)[::-1][:top_k]

X_all_raw = beta_mat[top_idx, :].T  # samples x top_k

# 4) PCA RAW
pcs_norm_raw, pcs_all_raw = fit_pca_on_normal_and_project(X_all_raw, normal_mask, N_PCS)
corr_pc1_age_raw = corr(pcs_norm_raw[:, 0], ages[normal_mask])

emb_raw = pd.DataFrame(pcs_all_raw, columns=[f"PC{i+1}_raw" for i in range(N_PCS)])
emb_raw[ID_COL] = samples
emb_raw["label"] = labels
emb_raw[AGE_COL] = ages
emb_raw.to_csv(os.path.join(OUTDIR, "pca_raw_embeddings.csv"), index=False)

apply_thesis_style(use_tex=True, legend_position="best", legend_outside=True)

plt.figure()
sns.scatterplot(
    data=emb_raw,
    x="PC1_raw",
    y="PC2_raw",
    hue="label",
    palette=palette_labels,
    size=AGE_COL,
    sizes=(20, 80),
    alpha=0.85,
    edgecolor="none"
)
plt.xlabel("PC1 (raw)")
plt.ylabel("PC2 (raw)")
place_legend(outside=True)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "pca_raw.pdf"))
plt.close()

print(f"GSE225845 — PCA raw (fit Normal), corr(PC1, DNAmAge)={corr_pc1_age_raw:.2f}")

# 5) CpG-wise age association in Normal (SUMS-based OLS)
Y = beta_norm.astype(np.float32)            # CpGs x n_norm
x = ages[normal_mask].astype(np.float64)    # n_norm
x2 = x * x

obs = ~np.isnan(Y)
n_obs = obs.sum(axis=1).astype(np.int64)
valid = n_obs >= MIN_OBS_NORMAL

sy  = np.nansum(Y, axis=1).astype(np.float64)
sy2 = np.nansum((Y * Y), axis=1).astype(np.float64)
sxy = np.nansum(Y * x[None, :], axis=1).astype(np.float64)
sx  = (obs * x[None, :]).sum(axis=1).astype(np.float64)
sx2 = (obs * x2[None, :]).sum(axis=1).astype(np.float64)

den = sx2 - (sx * sx) / np.maximum(n_obs, 1)
num = sxy - (sx * sy) / np.maximum(n_obs, 1)

b = np.full(Y.shape[0], np.nan, dtype=np.float64)
a = np.full(Y.shape[0], np.nan, dtype=np.float64)

good = valid & np.isfinite(den) & (den > 0)
b[good] = num[good] / den[good]
a[good] = (sy[good] - b[good] * sx[good]) / n_obs[good]

# p-values
df = (n_obs - 2).astype(np.float64)
t_stat = np.full(Y.shape[0], np.nan, dtype=np.float64)
SSE = np.full(Y.shape[0], np.nan, dtype=np.float64)
SSE[good] = (
    sy2[good]
    - 2.0 * a[good] * sy[good]
    - 2.0 * b[good] * sxy[good]
    + n_obs[good] * (a[good] ** 2)
    + 2.0 * a[good] * b[good] * sx[good]
    + (b[good] ** 2) * sx2[good]
)
sigma2 = np.full(Y.shape[0], np.nan, dtype=np.float64)
sigma2[good] = SSE[good] / df[good]
se_b = np.full(Y.shape[0], np.nan, dtype=np.float64)
se_b[good] = np.sqrt(sigma2[good] / den[good])
t_stat[good] = b[good] / se_b[good]

try:
    from scipy.stats import t as tdist
    pvals = np.full(Y.shape[0], np.nan, dtype=np.float64)
    pvals[good] = 2.0 * tdist.sf(np.abs(t_stat[good]), df=df[good])
except Exception:
    pvals = np.full(Y.shape[0], np.nan, dtype=np.float64)
    x_ = np.abs(t_stat[good]) / np.sqrt(2.0)
    Phi = 0.5 * (1.0 + np.erf(x_))
    pvals[good] = 2.0 * (1.0 - Phi)

qvals = bh_fdr(pvals)

assoc = pd.DataFrame({
    "cpg": cpg_index,
    "n_obs_normal": n_obs,
    "slope_per_year": b,
    "intercept": a,
    "t_stat": t_stat,
    "pval": pvals,
    "qval_fdr": qvals
}).set_index("cpg")
assoc.to_csv(os.path.join(OUTDIR, "age_association_normal_cpgwise.csv"))

# 6) Residualization ONLY for PCA features
a_top = a[top_idx].astype(np.float32)
b_top = b[top_idx].astype(np.float32)
ages_f = ages.astype(np.float32)

X_all_resid = X_all_raw.astype(np.float32) - a_top[None, :] - ages_f[:, None] * b_top[None, :]

# 7) PCA residualized
pcs_norm_res, pcs_all_res = fit_pca_on_normal_and_project(X_all_resid, normal_mask, N_PCS)
corr_pc1_age_res = corr(pcs_norm_res[:, 0], ages[normal_mask])

emb_res = pd.DataFrame(pcs_all_res, columns=[f"PC{i+1}_resid" for i in range(N_PCS)])
emb_res[ID_COL] = samples
emb_res["label"] = labels
emb_res[AGE_COL] = ages
emb_res.to_csv(os.path.join(OUTDIR, "pca_residual_embeddings.csv"), index=False)

apply_thesis_style(use_tex=True, legend_position="best", legend_outside=True)

plt.figure()
sns.scatterplot(
    data=emb_res,
    x="PC1_resid",
    y="PC2_resid",
    hue="label",
    palette=palette_labels,
    size=AGE_COL,
    sizes=(20, 80),
    alpha=0.85,
    edgecolor="none"
)
plt.xlabel("PC1 (residualized)")
plt.ylabel("PC2 (residualized)")
place_legend(outside=True)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "pca_residual.pdf"))
plt.close()

print(f"GSE225845 — PCA residualized, corr(PC1, DNAmAge)={corr_pc1_age_res:.2f}")

# 8) Summary
n_age_cpg = int((assoc["qval_fdr"] < 0.05).sum())
summary = pd.DataFrame([{
    "corr_PC1_raw_age_in_Normal": corr_pc1_age_raw,
    "corr_PC1_resid_age_in_Normal": corr_pc1_age_res,
    "n_samples_total": int(len(samples)),
    "n_samples_normal": int(normal_mask.sum()),
    "n_cpgs_total": int(beta_mat.shape[0]),
    "top_k_for_pca": int(top_k),
    "min_obs_normal_for_ols": int(MIN_OBS_NORMAL),
    "n_age_cpgs_FDR_0p05": n_age_cpg,
}])
summary.to_csv(os.path.join(OUTDIR, "run2_summary_metrics.csv"), index=False)

print("Saved outputs in:", OUTDIR)
print(summary.to_string(index=False))


Detected WIDE beta format (rows=samples, columns=CpGs).
Aligned samples: 253
Beta matrix shape (CpGs x samples): (747602, 253)


/tmp/ipykernel_54/3896855475.py:191: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


GSE225845 — PCA raw (fit Normal), corr(PC1, DNAmAge)=0.17
GSE225845 — PCA residualized, corr(PC1, DNAmAge)=0.00
Saved outputs in: ./gse225845_run2_age_horvath_outputs_optimized
 corr_PC1_raw_age_in_Normal  corr_PC1_resid_age_in_Normal  n_samples_total  n_samples_normal  n_cpgs_total  top_k_for_pca  min_obs_normal_for_ols  n_age_cpgs_FDR_0p05
                   0.166188                  4.657238e-08              253               113        747602          10000                      20                 5632


In [10]:
# SUMMARY OF AGE EFFECT ANALYSIS (CHRONOLOGICAL AGE, NORMAL-ONLY)
OUTDIR = "./gse225845_run1_age_true_outputs_optimized"

RAW_PATH   = os.path.join(OUTDIR, "pca_raw_embeddings.csv")
RES_PATH   = os.path.join(OUTDIR, "pca_residual_embeddings.csv")
ASSOC_PATH = os.path.join(OUTDIR, "age_association_normal_cpgwise.csv")

# Helpers
def corr_nan(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return np.nan
    a = a[ok] - np.mean(a[ok])
    b = b[ok] - np.mean(b[ok])
    denom = np.sqrt(np.sum(a*a) * np.sum(b*b))
    return np.nan if denom == 0 else float(np.sum(a*b) / denom)

def summarize_age_cpgs(assoc_df, q_col="qval_fdr"):
    qs = assoc_df[q_col].to_numpy(dtype=np.float64)
    out = {}
    for thr in [0.20, 0.10, 0.05, 0.01]:
        out[f"n_age_cpgs_{q_col}_lt_{thr}"] = int(np.nansum(qs < thr))
    return out

# Load
emb_raw = pd.read_csv(RAW_PATH)
emb_res = pd.read_csv(RES_PATH)
assoc   = pd.read_csv(ASSOC_PATH)

# sanity: columns
assert "age_at_surgery" in emb_raw.columns, "age_at_surgery missing in pca_raw_embeddings.csv"
assert "age_at_surgery" in emb_res.columns, "age_at_surgery missing in pca_residual_embeddings.csv"
assert "label" in emb_raw.columns and "label" in emb_res.columns, "label missing in embeddings"
assert "qval_fdr" in assoc.columns, "qval_fdr missing in age_association file"
assert "slope_per_year" in assoc.columns, "slope_per_year missing in age_association file"

# if assoc has CpG in index or column
if "cpg" in assoc.columns:
    assoc = assoc.set_index("cpg")
else:
    # if already saved with index, keep as-is
    if assoc.index.name is None:
        assoc.index.name = "cpg"

# 1 How many Age-CpGs 
counts = summarize_age_cpgs(assoc, q_col="qval_fdr")
print("=== Age-CpG counts (Normal-only association) ===")
for k, v in counts.items():
    print(f"{k}: {v}")

# 2 corr(age, PCs) in NORMAL 
NORMAL_LABEL = 0
raw_norm = emb_raw.loc[emb_raw["label"] == NORMAL_LABEL].copy()
res_norm = emb_res.loc[emb_res["label"] == NORMAL_LABEL].copy()

pc_cols_raw = [c for c in raw_norm.columns if c.startswith("PC") and c.endswith("_raw")]
pc_cols_res = [c for c in res_norm.columns if c.startswith("PC") and c.endswith("_resid")]

pc_cols_raw = sorted(pc_cols_raw, key=lambda s: int(s.split("PC")[1].split("_")[0]))
pc_cols_res = sorted(pc_cols_res, key=lambda s: int(s.split("PC")[1].split("_")[0]))

max_pcs = min(5, len(pc_cols_raw), len(pc_cols_res))

corr_rows = []
for i in range(max_pcs):
    pc_r = pc_cols_raw[i]
    pc_e = pc_cols_res[i]
    corr_rows.append({
        "PC": f"PC{i+1}",
        "corr_raw_age_in_Normal": corr_nan(raw_norm[pc_r], raw_norm["age_at_surgery"]),
        "corr_resid_age_in_Normal": corr_nan(res_norm[pc_e], res_norm["age_at_surgery"]),
    })

corr_df = pd.DataFrame(corr_rows)
print("\n=== Correlation between age and PCs (Normal only) ===")
print(corr_df.to_string(index=False))

corr_df.to_csv(os.path.join(OUTDIR, "corr_age_vs_pcs_normal.csv"), index=False)

# 3 Effect size summaries for Age-CpGs 
age_cpgs_05 = assoc.loc[assoc["qval_fdr"] < 0.05].copy()
print("\n=== Effect sizes (slope per year) for Age-CpGs (FDR<0.05) ===")
if age_cpgs_05.shape[0] == 0:
    print("No CpGs at FDR<0.05.")
else:
    slopes = age_cpgs_05["slope_per_year"].to_numpy(dtype=np.float64)
    abs_slopes = np.abs(slopes[np.isfinite(slopes)])
    print("n:", abs_slopes.size)
    print("abs(slope) quantiles (beta units / year):",
          np.quantile(abs_slopes, [0.50, 0.75, 0.90, 0.95, 0.99]))
    print("max abs(slope):", float(np.max(abs_slopes)))

# Save a compact table of top Age-CpGs by |t| (or by smallest q)
top_by_q = assoc.sort_values("qval_fdr", ascending=True).head(200)
top_by_q.to_csv(os.path.join(OUTDIR, "top200_age_cpgs_by_qval.csv"))

top_by_abs_t = assoc.assign(abs_t=np.abs(assoc["t_stat"])).sort_values("abs_t", ascending=False).head(200)
top_by_abs_t.drop(columns=["abs_t"]).to_csv(os.path.join(OUTDIR, "top200_age_cpgs_by_abs_t.csv"))

# 5 One-line "result" string you can paste into notes 
result_line = (
    f"GSE225845 (Normal-only fit): corr(age, PC1) raw={corr_df.loc[0,'corr_raw_age_in_Normal']:.3f}, "
    f"resid={corr_df.loc[0,'corr_resid_age_in_Normal']:.3f}; "
    f"Age-CpGs (FDR<0.05)={counts['n_age_cpgs_qval_fdr_lt_0.05']}"
)
print("\n=== One-line summary ===")
print(result_line)
with open(os.path.join(OUTDIR, "run1_one_line_summary.txt"), "w") as f:
    f.write(result_line + "\n")


=== Age-CpG counts (Normal-only association) ===
n_age_cpgs_qval_fdr_lt_0.2: 2873
n_age_cpgs_qval_fdr_lt_0.1: 962
n_age_cpgs_qval_fdr_lt_0.05: 533
n_age_cpgs_qval_fdr_lt_0.01: 217

=== Correlation between age and PCs (Normal only) ===
 PC  corr_raw_age_in_Normal  corr_resid_age_in_Normal
PC1               -0.033114              4.360097e-08
PC2                0.001541             -4.723215e-08
PC3               -0.070863             -9.994525e-08
PC4               -0.242033              5.616517e-08
PC5               -0.012772              2.234485e-07

=== Effect sizes (slope per year) for Age-CpGs (FDR<0.05) ===
n: 533
abs(slope) quantiles (beta units / year): [0.00132439 0.00203426 0.00267613 0.00304139 0.0038901 ]
max abs(slope): 0.0045991568225476

=== One-line summary ===
GSE225845 (Normal-only fit): corr(age, PC1) raw=-0.033, resid=0.000; Age-CpGs (FDR<0.05)=533


## 03- Check to confirm the study

In [11]:
# SUMMARY OF AGE EFFECT ANALYSIS (HORVATH DNAmAge, NORMAL-ONLY)
OUTDIR = "./gse225845_run2_age_horvath_outputs_optimized"

RAW_PATH   = os.path.join(OUTDIR, "pca_raw_embeddings.csv")
RES_PATH   = os.path.join(OUTDIR, "pca_residual_embeddings.csv")
ASSOC_PATH = os.path.join(OUTDIR, "age_association_normal_cpgwise.csv")

AGE_COL = "DNAmAge"
NORMAL_LABEL = 0

# Helpers
def corr_nan(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return np.nan
    a = a[ok] - np.mean(a[ok])
    b = b[ok] - np.mean(b[ok])
    denom = np.sqrt(np.sum(a*a) * np.sum(b*b))
    return np.nan if denom == 0 else float(np.sum(a*b) / denom)

def summarize_age_cpgs(assoc_df, q_col="qval_fdr"):
    qs = assoc_df[q_col].to_numpy(dtype=np.float64)
    out = {}
    for thr in [0.20, 0.10, 0.05, 0.01]:
        out[f"n_age_cpgs_{q_col}_lt_{thr}"] = int(np.nansum(qs < thr))
    return out

# Load
emb_raw = pd.read_csv(RAW_PATH)
emb_res = pd.read_csv(RES_PATH)
assoc   = pd.read_csv(ASSOC_PATH)

# sanity: columns
assert AGE_COL in emb_raw.columns, f"{AGE_COL} missing in pca_raw_embeddings.csv"
assert AGE_COL in emb_res.columns, f"{AGE_COL} missing in pca_residual_embeddings.csv"
assert "label" in emb_raw.columns and "label" in emb_res.columns, "label missing in embeddings"
assert "qval_fdr" in assoc.columns, "qval_fdr missing in age_association file"
assert "slope_per_year" in assoc.columns, "slope_per_year missing in age_association file"
assert "t_stat" in assoc.columns, "t_stat missing in age_association file"

# if assoc has CpG in index or column
if "cpg" in assoc.columns:
    assoc = assoc.set_index("cpg")
else:
    if assoc.index.name is None:
        assoc.index.name = "cpg"

# 1) How many Age-CpGs
counts = summarize_age_cpgs(assoc, q_col="qval_fdr")
print("=== Age-CpG counts (Normal-only association; age=DNAmAge) ===")
for k, v in counts.items():
    print(f"{k}: {v}")

# 2) corr(DNAmAge, PCs) in NORMAL
raw_norm = emb_raw.loc[emb_raw["label"] == NORMAL_LABEL].copy()
res_norm = emb_res.loc[emb_res["label"] == NORMAL_LABEL].copy()

pc_cols_raw = [c for c in raw_norm.columns if c.startswith("PC") and c.endswith("_raw")]
pc_cols_res = [c for c in res_norm.columns if c.startswith("PC") and c.endswith("_resid")]

pc_cols_raw = sorted(pc_cols_raw, key=lambda s: int(s.split("PC")[1].split("_")[0]))
pc_cols_res = sorted(pc_cols_res, key=lambda s: int(s.split("PC")[1].split("_")[0]))

max_pcs = min(5, len(pc_cols_raw), len(pc_cols_res))

corr_rows = []
for i in range(max_pcs):
    pc_r = pc_cols_raw[i]
    pc_e = pc_cols_res[i]
    corr_rows.append({
        "PC": f"PC{i+1}",
        "corr_raw_age_in_Normal": corr_nan(raw_norm[pc_r], raw_norm[AGE_COL]),
        "corr_resid_age_in_Normal": corr_nan(res_norm[pc_e], res_norm[AGE_COL]),
    })

corr_df = pd.DataFrame(corr_rows)
print("\n=== Correlation between DNAmAge and PCs (Normal only) ===")
print(corr_df.to_string(index=False))

corr_df.to_csv(os.path.join(OUTDIR, "corr_dnamage_vs_pcs_normal.csv"), index=False)

# 3) Effect size summaries for Age-CpGs
age_cpgs_05 = assoc.loc[assoc["qval_fdr"] < 0.05].copy()
print("\n=== Effect sizes (slope per year) for Age-CpGs (FDR<0.05) ===")
if age_cpgs_05.shape[0] == 0:
    print("No CpGs at FDR<0.05.")
else:
    slopes = age_cpgs_05["slope_per_year"].to_numpy(dtype=np.float64)
    abs_slopes = np.abs(slopes[np.isfinite(slopes)])
    print("n:", abs_slopes.size)
    print("abs(slope) quantiles (beta units / year):",
          np.quantile(abs_slopes, [0.50, 0.75, 0.90, 0.95, 0.99]))
    print("max abs(slope):", float(np.max(abs_slopes)))

# Save a compact table of top Age-CpGs by |t| (or by smallest q)
top_by_q = assoc.sort_values("qval_fdr", ascending=True).head(200)
top_by_q.to_csv(os.path.join(OUTDIR, "top200_age_cpgs_by_qval.csv"))

top_by_abs_t = assoc.assign(abs_t=np.abs(assoc["t_stat"])).sort_values("abs_t", ascending=False).head(200)
top_by_abs_t.drop(columns=["abs_t"]).to_csv(os.path.join(OUTDIR, "top200_age_cpgs_by_abs_t.csv"))

# 4) One-line "result" string you can paste into notes
result_line = (
    f"GSE225845 (Normal-only fit; age=DNAmAge): corr(age, PC1) raw={corr_df.loc[0,'corr_raw_age_in_Normal']:.3f}, "
    f"resid={corr_df.loc[0,'corr_resid_age_in_Normal']:.3f}; "
    f"Age-CpGs (FDR<0.05)={counts['n_age_cpgs_qval_fdr_lt_0.05']}"
)
print("\n=== One-line summary ===")
print(result_line)
with open(os.path.join(OUTDIR, "run2_one_line_summary.txt"), "w") as f:
    f.write(result_line + "\n")


=== Age-CpG counts (Normal-only association; age=DNAmAge) ===
n_age_cpgs_qval_fdr_lt_0.2: 32875
n_age_cpgs_qval_fdr_lt_0.1: 12640
n_age_cpgs_qval_fdr_lt_0.05: 5632
n_age_cpgs_qval_fdr_lt_0.01: 1225

=== Correlation between DNAmAge and PCs (Normal only) ===
 PC  corr_raw_age_in_Normal  corr_resid_age_in_Normal
PC1                0.166188              1.789946e-08
PC2                0.098656             -1.025964e-07
PC3               -0.110476             -2.049238e-07
PC4               -0.140775             -2.635844e-07
PC5               -0.100221              3.736171e-07

=== Effect sizes (slope per year) for Age-CpGs (FDR<0.05) ===
n: 5632
abs(slope) quantiles (beta units / year): [0.00175085 0.00278046 0.00359601 0.00406616 0.00501634]
max abs(slope): 0.0091524754823453

=== One-line summary ===
GSE225845 (Normal-only fit; age=DNAmAge): corr(age, PC1) raw=0.166, resid=0.000; Age-CpGs (FDR<0.05)=5632


In [12]:
# CHECK 1 — DNAmAge vs chronological age (Normal only)
# paths 
PHENO_TRUE_AGE_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"
DNAMAGE_CSV_PATH    = "/kaggle/input/gse225845-horvathdnamage/Output_GSE225845_HorvathDNAmAge_normal_adj_only.csv"

ID_COL = "id_tissue"
AGE_TRUE_COL = "age_at_surgery"
AGE_EST_COL  = "DNAmAge"
NORMAL_LABEL = 0

# load true age
pheno = pd.read_parquet(PHENO_TRUE_AGE_PATH)[[ID_COL, "label", AGE_TRUE_COL]]
pheno = pheno.loc[pheno["label"] == NORMAL_LABEL].dropna()

# load DNAmAge
dnm = pd.read_csv(DNAMAGE_CSV_PATH).rename(columns={"SampleID": ID_COL})
dnm = dnm[[ID_COL, AGE_EST_COL]].dropna()

# merge (Normal only)
df = pheno.merge(dnm, on=ID_COL, how="inner")

# correlation
corr = np.corrcoef(df[AGE_TRUE_COL], df[AGE_EST_COL])[0, 1]

print("=== DNAmAge vs age_at_surgery (Normal only) ===")
print(f"n samples: {df.shape[0]}")
print(f"Pearson corr: {corr:.3f}")


=== DNAmAge vs age_at_surgery (Normal only) ===
n samples: 113
Pearson corr: 0.703


In [13]:
# CHECK 2 — Overlap Age-CpGs (age true vs DNAmAge)
# paths
ASSOC_TRUE_AGE_PATH = "./gse225845_run1_age_true_outputs_optimized/age_association_normal_cpgwise.csv"
ASSOC_DNAMAGE_PATH  = "./gse225845_run2_age_horvath_outputs_optimized/age_association_normal_cpgwise.csv"

FDR_THR = 0.05

# load
assoc_true = pd.read_csv(ASSOC_TRUE_AGE_PATH)
assoc_dna  = pd.read_csv(ASSOC_DNAMAGE_PATH)

# set index if needed
if "cpg" in assoc_true.columns:
    assoc_true = assoc_true.set_index("cpg")
if "cpg" in assoc_dna.columns:
    assoc_dna = assoc_dna.set_index("cpg")

# significant CpGs
set_true = set(assoc_true.index[assoc_true["qval_fdr"] < FDR_THR])
set_dna  = set(assoc_dna.index[assoc_dna["qval_fdr"] < FDR_THR])

# overlap
inter = set_true & set_dna

print("=== Age-CpG overlap (Normal only, FDR < 0.05) ===")
print(f"Age true CpGs: {len(set_true)}")
print(f"DNAmAge CpGs:  {len(set_dna)}")
print(f"Overlap:       {len(inter)}")

if len(set_true) > 0:
    print(f"Overlap / age true: {len(inter) / len(set_true):.2%}")
if len(set_dna) > 0:
    print(f"Overlap / DNAmAge:  {len(inter) / len(set_dna):.2%}")


=== Age-CpG overlap (Normal only, FDR < 0.05) ===
Age true CpGs: 533
DNAmAge CpGs:  5632
Overlap:       253
Overlap / age true: 47.47%
Overlap / DNAmAge:  4.49%


## 04- Bin division
The division of bin adopted is shown in the following table.

| Bin | DNAmAge | Interpretation |
| --- | -------------- | --------------- |
| 1   | ≤ 40           | young           |
| 2   | 41 – 52        | peri-menopausal |
| 3   | 53 – 64        | post-menopausal |
| 4   | ≥ 65           | elderly         |


In [14]:
# CONVERSION OF AGE INTO BIN - 287331
# INPUT / OUTPUT
IN_PATH  = "/kaggle/input/gse287331-horvathdnamage/Output_GSE287331_HorvathDNAmAge_normal_adj_only.csv"
OUT_PATH = "Output_GSE287331_HorvathDNAmAge_normal_adj_only_with_bins.csv"

# LOAD
df = pd.read_csv(IN_PATH)

# sanity check
assert {"SampleID", "DNAmAge"}.issubset(df.columns), \
    f"Unexpected columns: {df.columns.tolist()}"

# DEFINE AGE BINS via Biology of breast cancer in young women - Hatem A Azim Jr1 and Ann H Partridge
def age_to_bin(age):
    if age <= 40:
        return 1
    elif age <= 52:
        return 2
    elif age <= 64:
        return 3
    else:
        return 4

df["age_bin"] = df["DNAmAge"].apply(age_to_bin).astype(int)

# optional: human-readable label (utile per plot / tesi)
df["age_bin_label"] = pd.cut(
    df["DNAmAge"],
    bins=[-float("inf"), 40, 52, 64, float("inf")],
    labels=["≤40", "41–52", "53–64", "≥65"]
)

# SAVE
df.to_csv(OUT_PATH, index=False)

print("Saved file with age bins to:")
print(OUT_PATH)

print("\nBin counts:")
print(df["age_bin"].value_counts().sort_index())


Saved file with age bins to:
Output_GSE287331_HorvathDNAmAge_normal_adj_only_with_bins.csv

Bin counts:
age_bin
1      5
2    124
3    101
4     12
Name: count, dtype: int64


In [15]:
# CONVERSION OF AGE INTO BIN - 69914
# INPUT / OUTPUT
IN_PATH  = "/kaggle/input/gse69914-horvathdnamage/Output_GSE69914_HorvathDNAmAge_normal_adj_only.csv"
OUT_PATH = "Output_GSE69914_HorvathDNAmAge_normal_adj_only_with_bins.csv"

# LOAD
df = pd.read_csv(IN_PATH)

# sanity check
assert {"SampleID", "DNAmAge"}.issubset(df.columns), \
    f"Unexpected columns: {df.columns.tolist()}"

# DEFINE AGE BINS via Biology of breast cancer in young women - Hatem A Azim Jr1 and Ann H Partridge
def age_to_bin(age):
    if age <= 40:
        return 1
    elif age <= 52:
        return 2
    elif age <= 64:
        return 3
    else:
        return 4

df["age_bin"] = df["DNAmAge"].apply(age_to_bin).astype(int)

# optional: human-readable label (utile per plot / tesi)
df["age_bin_label"] = pd.cut(
    df["DNAmAge"],
    bins=[-float("inf"), 40, 52, 64, float("inf")],
    labels=["≤40", "41–52", "53–64", "≥65"]
)

# SAVE
df.to_csv(OUT_PATH, index=False)

print("Saved file with age bins to:")
print(OUT_PATH)

print("\nBin counts:")
print(df["age_bin"].value_counts().sort_index())


Saved file with age bins to:
Output_GSE69914_HorvathDNAmAge_normal_adj_only_with_bins.csv

Bin counts:
age_bin
1     8
2    27
3    42
4    15
Name: count, dtype: int64


In [17]:
# ADD BIN INTO PHENO FILE - 225845
# PATHS
PHENO_PARQUET_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"
OUT_PHENO_PATH     = "pheno_GSE225845_with_true_age_bins.parquet"

ID_COL    = "id_tissue"
LABEL_COL = "label"
AGE_COL   = "age_at_surgery"

# LOAD
pheno = pl.read_parquet(PHENO_PARQUET_PATH)

# sanity checks
need = [ID_COL, LABEL_COL, AGE_COL]
missing = [c for c in need if c not in pheno.columns]
if missing:
    raise ValueError(f"Missing columns in pheno: {missing}. Available: {pheno.columns}")

# ensure types
pheno = pheno.with_columns([
    pl.col(ID_COL).cast(pl.Utf8),
    pl.col(LABEL_COL).cast(pl.Int64),
    pl.col(AGE_COL).cast(pl.Float64),
])

# BINNING RULES (Azim-like)
pheno_out = pheno.with_columns(
    pl.when(pl.col(LABEL_COL) == 2)
      .then(pl.lit(None))
      .otherwise(
          pl.when(pl.col(AGE_COL).is_null())
            .then(pl.lit(None))
            .when(pl.col(AGE_COL) <= 40)
            .then(pl.lit(1))
            .when((pl.col(AGE_COL) >= 41) & (pl.col(AGE_COL) <= 52))
            .then(pl.lit(2))
            .when((pl.col(AGE_COL) >= 53) & (pl.col(AGE_COL) <= 64))
            .then(pl.lit(3))
            .when(pl.col(AGE_COL) >= 65)
            .then(pl.lit(4))
            .otherwise(pl.lit(None))
      )
      .cast(pl.Int64)
      .alias("age_bin")
)

# SAVE
pheno_out.write_parquet(OUT_PHENO_PATH)

print("Saved updated pheno with true-age bins to:")
print(OUT_PHENO_PATH)

print("\nAge-bin counts by label (Tumor should have NULL bins):")
print(
    pheno_out
    .group_by([LABEL_COL, "age_bin"])
    .count()
    .sort([LABEL_COL, "age_bin"])
)

print("\nSanity: how many Tumor have non-null bins? (should be 0)")
print(
    pheno_out
    .filter(pl.col(LABEL_COL) == 2)
    .select(pl.col("age_bin").is_not_null().sum().alias("tumor_bins_not_null"))
)


Saved updated pheno with true-age bins to:
pheno_GSE225845_with_true_age_bins.parquet

Age-bin counts by label (Tumor should have NULL bins):
shape: (9, 3)
┌───────┬─────────┬───────┐
│ label ┆ age_bin ┆ count │
│ ---   ┆ ---     ┆ ---   │
│ i64   ┆ i64     ┆ u32   │
╞═══════╪═════════╪═══════╡
│ 0     ┆ 1       ┆ 57    │
│ 0     ┆ 2       ┆ 42    │
│ 0     ┆ 3       ┆ 13    │
│ 0     ┆ 4       ┆ 1     │
│ 1     ┆ 1       ┆ 20    │
│ 1     ┆ 2       ┆ 42    │
│ 1     ┆ 3       ┆ 36    │
│ 1     ┆ 4       ┆ 42    │
│ 2     ┆ null    ┆ 224   │
└───────┴─────────┴───────┘

Sanity: how many Tumor have non-null bins? (should be 0)
shape: (1, 1)
┌─────────────────────┐
│ tumor_bins_not_null │
│ ---                 │
│ u32                 │
╞═════════════════════╡
│ 0                   │
└─────────────────────┘


/tmp/ipykernel_54/2184414904.py:57: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  .count()


In [20]:
# ADD BIN INTO PHENO FILE - 287331
# PATHS
PHENO_PARQUET_PATH = "/kaggle/input/1-pheno-gse287331-parquet/pheno_GSE287331.parquet"
AGE_BIN_CSV_PATH   = "Output_GSE287331_HorvathDNAmAge_normal_adj_only_with_bins.csv"

OUT_PHENO_PATH     = "pheno_GSE287331_with_age_bin.parquet"

# LOAD PHENO
pheno = pl.read_parquet(PHENO_PARQUET_PATH)

# sanity
assert "id_tissue" in pheno.columns
assert "label" in pheno.columns

# LOAD AGE BIN FILE
age_bins = pl.read_csv(AGE_BIN_CSV_PATH).select([
    pl.col("SampleID").cast(pl.Utf8).alias("id_tissue"),
    pl.col("age_bin").cast(pl.Int64)
])

# LEFT JOIN (keep all pheno)
pheno_out = pheno.join(
    age_bins,
    on="id_tissue",
    how="left"
)

# SET age_bin = NULL for Tumor (label == 2)
pheno_out = pheno_out.with_columns(
    pl.when(pl.col("label") == 2)
      .then(pl.lit(None))
      .otherwise(pl.col("age_bin"))
      .alias("age_bin")
)

# SAVE
pheno_out.write_parquet(OUT_PHENO_PATH)

print("Saved updated pheno with age_bin to:")
print(OUT_PHENO_PATH)

print("\nAge bin counts by label:")
print(
    pheno_out
    .group_by([LABEL_COL, "age_bin"])
    .count()
    .sort([LABEL_COL, "age_bin"])
)


Saved updated pheno with age_bin to:
pheno_GSE287331_with_age_bin.parquet

Age bin counts by label:
shape: (11, 3)
┌───────┬─────────┬───────┐
│ label ┆ age_bin ┆ count │
│ ---   ┆ ---     ┆ ---   │
│ i8    ┆ i64     ┆ u32   │
╞═══════╪═════════╪═══════╡
│ 0     ┆ 1       ┆ 4     │
│ 0     ┆ 2       ┆ 101   │
│ 0     ┆ 3       ┆ 71    │
│ 0     ┆ 4       ┆ 6     │
│ 1     ┆ 1       ┆ 1     │
│ …     ┆ …       ┆ …     │
│ 1     ┆ 3       ┆ 30    │
│ 1     ┆ 4       ┆ 6     │
│ 2     ┆ null    ┆ 69    │
│ 3     ┆ null    ┆ 67    │
│ 4     ┆ null    ┆ 68    │
└───────┴─────────┴───────┘


/tmp/ipykernel_54/2409130496.py:46: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  .count()


In [21]:
# ADD BIN INTO PHENO FILE - 69914
# PATHS
PHENO_PARQUET_PATH = "/kaggle/input/pheno-gse69914/pheno_GSE69914.parquet"
AGE_BIN_CSV_PATH   = "Output_GSE69914_HorvathDNAmAge_normal_adj_only_with_bins.csv"

OUT_PHENO_PATH     = "pheno_GSE69914_with_age_bin.parquet"

# LOAD PHENO
pheno = pl.read_parquet(PHENO_PARQUET_PATH)

# sanity
assert "id_tissue" in pheno.columns
assert "label" in pheno.columns

# LOAD AGE BIN FILE
age_bins = pl.read_csv(AGE_BIN_CSV_PATH).select([
    pl.col("SampleID").cast(pl.Utf8).alias("id_tissue"),
    pl.col("age_bin").cast(pl.Int64)
])

# LEFT JOIN (keep all pheno)
pheno_out = pheno.join(
    age_bins,
    on="id_tissue",
    how="left"
)

# SET age_bin = NULL for Tumor (label == 2)
pheno_out = pheno_out.with_columns(
    pl.when(pl.col("label") == 2)
      .then(pl.lit(None))
      .otherwise(pl.col("age_bin"))
      .alias("age_bin")
)

# SAVE
pheno_out.write_parquet(OUT_PHENO_PATH)

print("Saved updated pheno with age_bin to:")
print(OUT_PHENO_PATH)

print("\nAge bin counts by label:")
print(
    pheno_out
    .group_by([LABEL_COL, "age_bin"])
    .count()
    .sort([LABEL_COL, "age_bin"])
)


Saved updated pheno with age_bin to:
pheno_GSE69914_with_age_bin.parquet

Age bin counts by label:
shape: (11, 3)
┌───────┬─────────┬───────┐
│ label ┆ age_bin ┆ count │
│ ---   ┆ ---     ┆ ---   │
│ i64   ┆ i64     ┆ u32   │
╞═══════╪═════════╪═══════╡
│ 0     ┆ 1       ┆ 7     │
│ 0     ┆ 2       ┆ 13    │
│ 0     ┆ 3       ┆ 22    │
│ 0     ┆ 4       ┆ 8     │
│ 1     ┆ 1       ┆ 1     │
│ …     ┆ …       ┆ …     │
│ 1     ┆ 3       ┆ 20    │
│ 1     ┆ 4       ┆ 7     │
│ 2     ┆ null    ┆ 305   │
│ 3     ┆ null    ┆ 7     │
│ 4     ┆ null    ┆ 3     │
└───────┴─────────┴───────┘


/tmp/ipykernel_54/3406792094.py:46: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  .count()


In [8]:
# VISUALIZATION OF BIN DISTRIBUTION (BARPLOT ONLY, STANDARD LEGEND)
# CONFIG
DATA_PATHS = {
    "GSE69914_pheno":  "/kaggle/input/pheno-gse69914-with-age-bin/pheno_GSE69914_with_age_bin.parquet",
    "GSE287331_pheno": "/kaggle/input/pheno-gse287331-with-age-bin/pheno_GSE287331_with_age_bin.parquet",
    "GSE225845_pheno": "/kaggle/input/pheno-gse225845-with-true-age-bins/pheno_GSE225845_with_true_age_bins.parquet",
}

OUTDIR = "./age_bins_plots"
os.makedirs(OUTDIR, exist_ok=True)

BIN_COL = "age_bin"

BIN_ORDER = [1, 2, 3, 4]
BIN_LABELS = {
    1: "young",
    2: "peri-menopausal",
    3: "post-menopausal",
    4: "elderly",
}
BIN_RANGES = {
    1: r"$\leq 40$",
    2: r"$41$--$52$",
    3: r"$53$--$64$",
    4: r"$\geq 65$",
}

# LOAD
pheno_69914  = pl.read_parquet(DATA_PATHS["GSE69914_pheno"]).to_pandas()
pheno_287331 = pl.read_parquet(DATA_PATHS["GSE287331_pheno"]).to_pandas()
pheno_225845 = pl.read_parquet(DATA_PATHS["GSE225845_pheno"]).to_pandas()

DATASETS = {
    "GSE69914":  pheno_69914,
    "GSE287331": pheno_287331,
    "GSE225845": pheno_225845,
}

# HELPERS
def _normalize_bin_series(s: pd.Series) -> pd.Series:
    if s.dtype.kind in "iu":
        return s.astype(int)

    s_num = pd.to_numeric(s, errors="coerce")
    if s_num.notna().sum() > 0:
        return s_num.astype("Int64")

    inv = {v.lower(): k for k, v in BIN_LABELS.items()}
    return s.astype(str).str.strip().str.lower().map(inv).astype("Int64")


def plot_bin_counts_only(df: pd.DataFrame, dataset_name: str):
    apply_thesis_style()

    if BIN_COL not in df.columns:
        raise ValueError(f"[{dataset_name}] Missing column '{BIN_COL}'")

    bins_norm = _normalize_bin_series(df[BIN_COL]).dropna()

    counts = {b: int((bins_norm == b).sum()) for b in BIN_ORDER}

    x = np.arange(len(BIN_ORDER))
    y = np.array([counts[b] for b in BIN_ORDER], dtype=int)

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    cmap = plt.get_cmap("viridis")

    colors = [cmap(i / max(1, len(BIN_ORDER) - 1)) for i in range(len(BIN_ORDER))]
    ax.bar(x, y, color=colors, edgecolor="none")

    ax.set_xticks(x)
    ax.set_xticklabels([BIN_LABELS[b] for b in BIN_ORDER])
    ax.set_ylabel("Number of samples")
    #ax.set_title(f"{dataset_name} — Sample count across age bins")
    ax.grid(axis="y", alpha=0.25)

    # ---- Standard legend (handled by apply_thesis_style)
    legend_handles = [
        Patch(facecolor=colors[i], edgecolor="none",
              label=f"{BIN_LABELS[b]} = {BIN_RANGES[b]}")
        for i, b in enumerate(BIN_ORDER)
    ]
    ax.legend(handles=legend_handles)

    fig.tight_layout()

    outpath = f"{OUTDIR}/{dataset_name}_bin_counts.pdf"
    fig.savefig(outpath)
    plt.close(fig)

    print(f"[✓] Saved: {outpath}")

for name, pheno in DATASETS.items():
    plot_bin_counts_only(pheno, name)


FileNotFoundError: No such file or directory (os error 2): /kaggle/input/pheno-gse287331-with-age-bin/pheno_GSE287331_with_age_bin.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'
